In [7]:
print("Test Run: Stem cell project notebook is working!")

Test Run: Stem cell project notebook is working!


### Doing Series GSE130340
The Experiment is about watching stem cells changing into another cell type and what genes become more or less active!
Starts with Human embryonic stem cells (hESCs) -> cells that are very flexible and can develop into many different kinds of cells
What this experiment GSE130340 is about: They gave the stem cells (hESCs) a signal that causes them to start becoming endoderm cells
What is endoderm cells? It's an early developmental cell layer that later helps form organs like liver/pancreas/lungs/digestive system
Summary embryonic stem cells -> differentiation -> Endoderm cell
They measured gene expression before and after that change, the main question is:
### When a stem cell starts becoming endoderm, which genes turn up and which genes turn down, which is what we will be analyzing today!
A few infomation notes on how to use the NCBI website (since it might look confusing but could be good notes/info for anyone!) https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE130340&utm_source=chatgpt.com
- **Query DataSets** for GSE130340 searches the GEO's dataset records related to this study, it can help find related entries connected to it, or even a more structured view of the experiment (we don't need this currently) basically can show same study/data with organized dataset-focused view instead of orginal study record page
- **GEO2R** is a web tool inside GEO that compares groups for you, example you should tell the tool group 1 = stem cells and group 2 = endoderm cells, then it will help which genes are expressed differently between those two groups. Very useful and pretty common tool, people use it because it's fast, no coding needed, good for checking interesting differences and could help comparing your own analysis! Downside is, you have less control on the analysis is done, no pratice in organizing data yourself or cleaning it, can be hard to customize... it goes on, but it's a good quick and convenient tool but with python you would be able to have more control.
- **The Download RNA-seq counts** is a table how much RNA was detected for each gene in each sample, example gene A has 120 fragments/reads associated with gene A, gene b has 540 count etc. The higher RNA-seq count = higher gene expression. Those counts come from the RNA-sequencing reads that were matched to genes, for this study the count files let us compare stem cell camples vs endoderm samples to see which genes are more or less active, There are usually different versions of counts, raw counts is closer to the orginal sequencing results while normalized counts are adjusted so samples are easier/fairer to compare. for this project we will be using the normalized count first! So basically RNA helps show what kind of cells it is and what it's doing.
- **Platforms (1): GPL18573 Illumina NextSeq 500** is the lab equipment/technology used to generate data (machine/platform)
- **Samples (6)** are the "6" samples in this case that has all the data, the biological sample, the actual physical sample from the experiment (like a batch of the stem cells or batch of endoderm cells for this experiement), so think of this as the orginal data before it got cleaned up into the RNA-seq count file
- **Supplementary files** are just additional supporting files attached to the study which could be extra data/results, which we won't be touching this
## So the only file we are downloading today is the RNA-Seq count data:
Submitter-supplied data → Series supplementary data → DESeq2 normalized counts

We are using the submitter-supplied DESeq2 normalized counts because it comes from the researchers’ original RNA-seq experiment but has already been processed enough to make the samples easier to compare. It is not fully raw sequencing data, but it is still close to the original gene-count data and is a good starting point for analysis.

In [8]:
import pandas as pd

gene_df = pd.read_csv(                                              # loading data file into a datafrom called gene_df
    "../data/raw/GSE130340_DESeq2_normalizedCounts.txt",            # location of our data file
    sep="\t"                                                        # tells pandas the columns are separated by tabs (helps to make it look cleaning into a actual column)
)

gene_df.head()                                                      # shows first 5 rows (0-4)

,GeneName,YFR-1,YFR-2,YFR-3,YFR-4,YFR-5,YFR-6
0,A1BG,8.023728,7.743835,8.065400,8.002750,7.692653,7.507645
1,A1BG-AS1,3.888766,3.915868,3.890196,4.215000,4.164389,4.240091
2,A1CF,2.012271,1.917230,2.004119,1.876895,1.894273,2.012169
3,A2M,0.367412,0.400001,0.362494,0.397569,0.391874,0.407677
4,A2M-AS1,2.688855,2.478498,2.609547,2.764179,2.702491,2.724217


We are seeing 6 samples (YFR-1, YFR-2, etc.) and multiple genes in the GeneName column. Each row represents one unique gene, and the numbers across the 6 samples show how much that gene is being expressed in each sample. The numbers show the normalized RNA count for each gene in each sample. Higher numbers usually mean the gene is being expressed more.

Now that we have a understanding of how our table works, we should ask: **Which genes change the most between the two cell groups?**
For this study we want to have a understanding what sample belong to which group, compare gene expression between the groups, find genes that go up/down a lot, make graphs like heatmaps/boxplots, and study what those genes actually do biologically. (BASICALLY A BIG PUZZLE MAP :D)

But lets understand our data one step more

In [9]:
gene_df.shape # shows the number of rows and columns in the data frame

(27317, 7)

In [10]:
gene_df.columns # shows the column names in the data frame

Index(['GeneName', 'YFR-1', 'YFR-2', 'YFR-3', 'YFR-4', 'YFR-5', 'YFR-6'], dtype='str')

In [11]:
gene_df.isnull().sum() # shows the number of missing values in each column

GeneName    0
YFR-1       0
YFR-2       0
YFR-3       0
YFR-4       0
YFR-5       0
YFR-6       0
dtype: int64

isnull() checks if anything is missing while .sum() counts how many missing values is in each column.
So each column says 0 meaning there is no missing values, and int64 means pandas is storing those 0 counts as whole numbers.

#### let's try to figure out what YFR sample belongs to stem cells or endodern cells 

In [12]:
gene_df.columns
gene_df.head(2)

,GeneName,YFR-1,YFR-2,YFR-3,YFR-4,YFR-5,YFR-6
0,A1BG,8.023728,7.743835,8.065400,8.00275,7.692653,7.507645
1,A1BG-AS1,3.888766,3.915868,3.890196,4.21500,4.164389,4.240091


In [ ]:
with open("../data/raw/GSE130340_DESeq2_normalizedCounts.txt", "r") as file:     # opening files in read mode
    for i in range(10):                                                          # repeating the next line 10 times
        print(file.readline())                                                   # while printing it for us to see the first 10 lines of the file

GeneName	YFR-1	YFR-2	YFR-3	YFR-4	YFR-5	YFR-6

A1BG	8.023727949	7.743835355	8.065400408	8.002750008	7.692653289	7.507644545

A1BG-AS1	3.888765837	3.915868466	3.890196231	4.21499998	4.164389003	4.240091351

A1CF	2.012271195	1.91723028	2.004118793	1.876894935	1.894273356	2.012168695

A2M	0.367412416	0.400000925	0.362493601	0.397569117	0.39187385	0.407676889

A2M-AS1	2.688854741	2.478498131	2.609546805	2.764178805	2.702490786	2.724217424

A2ML1	7.865601006	7.818958626	7.810546901	6.912164513	6.843541089	6.848734853

A2MP1	-1.97019103	-1.971748815	-1.972327491	-1.955909602	-1.973458166	-1.970196562

A3GALT2	-1.970117917	-1.956476418	-1.972089249	-1.972019199	-1.958053651	-1.970123022

A4GALT	8.915196446	8.640641087	8.797044869	8.376770314	8.312136025	8.353819439



Have to check website because there is no labels on what is an stem/endoderm cell, which is common for it to be seprated, so we will have to download the family.soft file, the file is mainly metadata meaning they are labels/explanations for those measurements we are looking at.

Also .soft file means it's already structured text with labels so they usually have ^SAMPLE and !Sample_title, you can verify this if you control  +  F in the family soft file

In [30]:
with open("../data/raw/GSE130340_family.soft", "r") as file:
    for line in file:
        if "^SAMPLE" in line:
            print(line)

^SAMPLE = GSM3736416

^SAMPLE = GSM3736417

^SAMPLE = GSM3736418

^SAMPLE = GSM3736419

^SAMPLE = GSM3736420

^SAMPLE = GSM3736421



In [ ]:
with open("../data/raw/GSE130340_family.soft", "r") as file:
    for line in file:
        if "!Sample_title" in line: # Basically control + F to find the line that has the sample title in it
            print(line)

!Sample_title = ES cells 1

!Sample_title = ES cells 2

!Sample_title = ES cells 3

!Sample_title = endoderm cells 1

!Sample_title = endoderm cells 2

!Sample_title = endoderm cells 3



In [32]:
with open("../data/raw/GSE130340_family.soft", "r") as file:     # opening files in read mode
    for line in file:
        if "YFR" in line:
            print(line)

!Sample_description = YFR1

!Sample_description = YFR2

!Sample_description = YFR3

!Sample_description = YFR4

!Sample_description = YFR5

!Sample_description = YFR6



Ok now we can match up the column titles we had (YFR1 etc.) and match them up with the sample_title infomation, so YFR1-3 are ES cells and YFR4-6 are endooderm cells
Lets now create our two list!

In [35]:
stem_samples = ["YFR-1", "YFR-2", "YFR-3"] # embryonic stem cell samples
endoderm_samples = ["YFR-4", "YFR-5", "YFR-6"] # endoderm samples

# now lets calculate the average expression for each gene in the two groups

gene_df["stem_avg"] = gene_df[stem_samples].mean(axis=1) # axis=1 means we are calculating the mean across columns (i.e. for each row)
gene_df["endoderm_avg"] = gene_df[endoderm_samples].mean(axis=1)

gene_df.head() # let's look at the first 5 rows of the data frame to see the new columns we just added

,GeneName,YFR-1,YFR-2,YFR-3,YFR-4,YFR-5,YFR-6,stem_avg,endoderm_avg
0,A1BG,8.023728,7.743835,8.065400,8.002750,7.692653,7.507645,7.944321,7.734349
1,A1BG-AS1,3.888766,3.915868,3.890196,4.215000,4.164389,4.240091,3.898277,4.206493
2,A1CF,2.012271,1.917230,2.004119,1.876895,1.894273,2.012169,1.977873,1.927779
3,A2M,0.367412,0.400001,0.362494,0.397569,0.391874,0.407677,0.376636,0.399040
4,A2M-AS1,2.688855,2.478498,2.609547,2.764179,2.702491,2.724217,2.592300,2.730296


Now we want to measure how much the expression changed

In [ ]:
gene_df["expression_change"] = gene_df["endoderm_avg"] - gene_df["stem_avg"] 
# our equation for the change in expression between the two groups (into a new column called expression_change)

gene_df.head()
# so what we want to see if there is a postive numner, its higher in endoderm, if its negative, its higher in stem cells

,GeneName,YFR-1,YFR-2,YFR-3,YFR-4,YFR-5,YFR-6,stem_avg,endoderm_avg,expression_change
0,A1BG,8.023728,7.743835,8.065400,8.002750,7.692653,7.507645,7.944321,7.734349,-0.209972
1,A1BG-AS1,3.888766,3.915868,3.890196,4.215000,4.164389,4.240091,3.898277,4.206493,0.308217
2,A1CF,2.012271,1.917230,2.004119,1.876895,1.894273,2.012169,1.977873,1.927779,-0.050094
3,A2M,0.367412,0.400001,0.362494,0.397569,0.391874,0.407677,0.376636,0.399040,0.022404
4,A2M-AS1,2.688855,2.478498,2.609547,2.764179,2.702491,2.724217,2.592300,2.730296,0.137996
